In [54]:
%pip install pandas numpy matplotlib scipy mpmath

Note: you may need to restart the kernel to use updated packages.


In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, chi2_contingency
import mpmath

In [56]:
CSV_PATH="data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [57]:
df=pd.read_csv(CSV_PATH)

raw_df = df.copy()
raw_df["TotalCharges"] = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")
raw_df.dropna(subset=["TotalCharges"], inplace=True)

display(raw_df.head())

display(raw_df.info())

display(raw_df.describe(include="all").T)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7032,7032,3186-AJIEK,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7032,2,Male,3549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7032.0,NaN,NaN,NaN,0.1624,0.368844,0.0,0.0,0.0,0.0,1.0
Partner,7032,2,No,3639,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7032,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7032.0,NaN,NaN,NaN,32.421786,24.54526,1.0,9.0,29.0,55.0,72.0
PhoneService,7032,2,Yes,6352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7032,3,No,3385,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7032,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7032,3,No,3497,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
CATEGORICAL_FEATURES=["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
 "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "Churn"]

for feature in CATEGORICAL_FEATURES:
    print(f"Unique Values for feature: {feature}\n{raw_df[feature].unique()}")
    print("="*100)

Unique Values for feature: gender
['Female' 'Male']
Unique Values for feature: SeniorCitizen
[0 1]
Unique Values for feature: Partner
['Yes' 'No']
Unique Values for feature: Dependents
['No' 'Yes']
Unique Values for feature: PhoneService
['No' 'Yes']
Unique Values for feature: MultipleLines
['No phone service' 'No' 'Yes']
Unique Values for feature: InternetService
['DSL' 'Fiber optic' 'No']
Unique Values for feature: OnlineSecurity
['No' 'Yes' 'No internet service']
Unique Values for feature: OnlineBackup
['Yes' 'No' 'No internet service']
Unique Values for feature: DeviceProtection
['No' 'Yes' 'No internet service']
Unique Values for feature: TechSupport
['No' 'Yes' 'No internet service']
Unique Values for feature: StreamingTV
['No' 'Yes' 'No internet service']
Unique Values for feature: StreamingMovies
['No' 'Yes' 'No internet service']
Unique Values for feature: Contract
['Month-to-month' 'One year' 'Two year']
Unique Values for feature: PaperlessBilling
['Yes' 'No']
Unique Values f

In [59]:
NUMERICAL_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]

raw_df[NUMERICAL_FEATURES].describe(percentiles=[.01, .1, .25, .5, .75, .90, .95, .99]).T

,count,mean,std,min,1%,10%,25%,50%,75%,90%,95%,99%,max
tenure,7032.0,32.421786,24.545260,1.00,1.0,2.00,9.0000,29.000,55.0000,69.000,72.0000,72.0000,72.00
MonthlyCharges,7032.0,64.798208,30.085974,18.25,19.2,20.05,35.5875,70.350,89.8625,102.645,107.4225,114.7345,118.75
TotalCharges,7032.0,2283.300441,2266.771362,18.80,19.9,84.60,401.4500,1397.475,3794.7375,5976.640,6923.5900,8039.8830,8684.80


In [62]:
# Convert 'Churn' column to numerical values
raw_df['Churn'] = raw_df['Churn'].map({'Yes': 1, 'No': 0})

# Initialize dictionaries to store results
ttest_results = {}
chi2_results = {}

# T-test for numerical features
for feature in NUMERICAL_FEATURES:
    churned = raw_df[raw_df['Churn'] == 1][feature]
    not_churned = raw_df[raw_df['Churn'] == 0][feature]
    
    t_stat, p_value = ttest_ind(churned, not_churned)
    ttest_results[feature] = {'t_stat': t_stat, 'p_value': p_value, 'significant': p_value < 0.05}

# Chi-square test for categorical features
for feature in CATEGORICAL_FEATURES:
    contingency_table = pd.crosstab(raw_df[feature], raw_df['Churn'])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    chi2_results[feature] = {'chi2': chi2, 'p_value': p_value, 'significant': p_value < 0.05}

# Output results with stylish prints
print("\n===== T-test Results for Numerical Features =====")
for feature, result in ttest_results.items():
    significance = "Significant ✅" if result['significant'] else "Not Significant ❌"
    print(f"Feature: {feature} | t-statistic = {result['t_stat']}, p-value = {result['p_value']} ({significance})")

print("\n===== Chi-square Test Results for Categorical Features =====")
for feature, result in chi2_results.items():
    significance = "Significant ✅" if result['significant'] else "Not Significant ❌"
    print(f"Feature: {feature} | chi-square = {result['chi2']:.4f}, p-value = {result['p_value']:.4f} ({significance})")


===== T-test Results for Numerical Features =====
Feature: tenure | t-statistic = -31.741289063447653, p-value = 9.437650217574845e-207 (Significant ✅)
Feature: MonthlyCharges | t-statistic = 16.47959313114872, p-value = 6.760843117980302e-60 (Significant ✅)
Feature: TotalCharges | t-statistic = -17.068827211220274, p-value = 4.876865689694505e-64 (Significant ✅)

===== Chi-square Test Results for Categorical Features =====
Feature: gender | chi-square = 0.4755, p-value = 0.4905 (Not Significant ❌)
Feature: SeniorCitizen | chi-square = 158.4408, p-value = 0.0000 (Significant ✅)
Feature: Partner | chi-square = 157.5032, p-value = 0.0000 (Significant ✅)
Feature: Dependents | chi-square = 186.3216, p-value = 0.0000 (Significant ✅)
Feature: PhoneService | chi-square = 0.8737, p-value = 0.3499 (Not Significant ❌)
Feature: MultipleLines | chi-square = 11.2715, p-value = 0.0036 (Significant ✅)
Feature: InternetService | chi-square = 728.6956, p-value = 0.0000 (Significant ✅)
Feature: OnlineS

In [65]:
# customer_id: drop
# gender: Convert to Binary
# SeniorCitizen: Nothing
# Partner: Convert to Binary
# Dependent: Convert to Binary
# Tenure: Standardization
# PhoneService: Convert to Binary
# MultipleLines: One-Hot Encoding
# InternetService: One-Hot Encoding
# OnlineSecurity: One-Hot Encoding
# DeviceProtection: One-Hot Encoding
# TechSupport: One-Hot Encoding
# StreamingTV: One-Hot Encoding
# StreamingMovies: One-Hot Encoding
# Contract: Ordinal-Encoding
# PaperlessBilling: Convert to Binary
# PaymentMethod: One-Hot Encoding
# AutomaticPayment: NEW FEATRURE
# MontlhyCharges: Standardization
# TotalCharges: Standardization
# Churn: 